In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, explained_variance_score, max_error, median_absolute_error
from src import calculate_metrics

## Preparacion de datos

In [6]:
df = pd.read_csv('../data/train/cleaned_train.csv')

y = df["Precio"]
X = df.drop(columns=["idx", "Tipo de carrocería", "Título", "Descripción", "Precio"])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

mean_encoded_cols = ["Marca", "Modelo", "Versión", "Color"]
for col in mean_encoded_cols:
    means = X_train.join(y_train).groupby(col)["Precio"].mean()
    X_train[col] = X_train[col].map(means)
    X_val[col] = X_val[col].map(means) 

one_hot_cols = ["Tipo de combustible", "Transmisión", "Con cámara de retroceso", "Tracción", "Tipo de vendedor"]

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols]) 

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[one_hot_cols] = encoder.fit_transform(X_train[one_hot_cols])
X_val[one_hot_cols] = encoder.transform(X_val[one_hot_cols])  

imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train)
X_val_imputed = imputer.transform(X_val)

X_train = pd.DataFrame(X_train_imputed, columns=X.columns, index=X_train.index)
X_val = pd.DataFrame(X_val_imputed, columns=X.columns, index=X_val.index)

X_train = pd.get_dummies(X_train, columns=one_hot_cols, dtype=np.float64)
X_val = pd.get_dummies(X_val, columns=one_hot_cols, dtype=np.float64)

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

Train: (11682, 39)
Val: (2921, 39)


## Base Line

In [9]:
import mlflow
from mlflow.models.signature import infer_signature

mlflow.set_experiment("Base Line")
mlflow.set_tracking_uri("http://localhost:5000")

In [10]:
reports = []

## Regresor lineal

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_train_lr = lr.predict(X_train)
y_pred_val_lr = lr.predict(X_val)
lr.coef_

In [13]:
lr_metrics = calculate_metrics(
    y_train, y_pred_train_lr, 
    y_val, y_pred_val_lr,
)

lr_metrics['name'] = "Linear Regression"
lr_metrics["model"] = lr
lr_metrics["params"] = None 

reports.append(lr_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   6.88e+13   |   9.11e+13
R²           |   0.8641     |   0.7986
MAE          |   4.45e+06   |   4.77e+06
EVS          |   0.8641     |   0.7988
Max Error    |   2.31e+08   |   2.86e+08
Median AE    |   2.76e+06   |   2.97e+06



## Random Forest

In [14]:
rf_params = {
    'n_estimators': 100,     
    'max_depth': None,        
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'random_state': 42
}

In [15]:
rf_reg = RandomForestRegressor(**rf_params)
rf_reg.fit(X_train, y_train)

y_pred_train_rf = rf_reg.predict(X_train)
y_pred_val_rf = rf_reg.predict(X_val)

In [16]:
rf_metrics = calculate_metrics(
    y_train, y_pred_train_rf,
    y_val, y_pred_val_rf,
)

rf_metrics["name"] = "Random Forest"
rf_metrics["model"] = rf_reg
rf_metrics["params"] = rf_params  

reports.append(rf_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   1.30e+13   |   5.63e+13
R²           |   0.9744     |   0.8756
MAE          |   1.19e+06   |   2.89e+06
EVS          |   0.9744     |   0.8756
Max Error    |   2.14e+08   |   2.75e+08
Median AE    |   5.51e+05   |   1.39e+06



## XGBoost

In [17]:
xgb_params = {
    'objective': 'reg:squarederror', 
    'n_estimators': 100,
    'max_depth': 4,
    'learning_rate': 0.1,
    'random_state': 42
}

In [18]:
xgb_reg = XGBRegressor(**xgb_params)
xgb_reg.fit(X_train, y_train)

y_pred_train_xgb = xgb_reg.predict(X_train)
y_pred_val_xgb = xgb_reg.predict(X_val)

In [19]:
xgb_metrics = calculate_metrics(
    y_train, y_pred_train_xgb,
    y_val, y_pred_val_xgb
)

xgb_metrics["name"] = "XGBoost"
xgb_metrics["model"] = xgb_reg
xgb_metrics["params"] = xgb_params  

reports.append(xgb_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   2.87e+13   |   5.10e+13
R²           |   0.9433     |   0.8871
MAE          |   2.90e+06   |   3.31e+06
EVS          |   0.9433     |   0.8872
Max Error    |   2.21e+08   |   2.24e+08
Median AE    |   1.81e+06   |   1.83e+06



## Red neuronal

In [20]:
mlp_params = {
    'hidden_layer_sizes': (100, 50),  
    'activation': 'relu',
    'solver': 'adam',
    'alpha': 0.0001,              
    'learning_rate': 'adaptive',
    'max_iter': 500,
    'random_state': 42
}

In [21]:
mlp_reg = MLPRegressor(**mlp_params)
mlp_reg.fit(X_train, y_train)

y_pred_train_mlp = mlp_reg.predict(X_train)
y_pred_val_mlp = mlp_reg.predict(X_val)

/home/elianaostro/miniconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [22]:
mlp_params_metrics = calculate_metrics(
    y_train, y_pred_train_mlp,
    y_val, y_pred_val_mlp
)

mlp_params_metrics["name"] = "MLP"
mlp_params_metrics["model"] = mlp_reg
mlp_params_metrics["params"] = mlp_params  

reports.append(mlp_params_metrics)

Metric       |    Train     |  Validation
-------------|--------------|---------------
MSE          |   8.19e+13   |   8.89e+13
R²           |   0.8382     |   0.8034
MAE          |   4.95e+06   |   5.07e+06
EVS          |   0.8382     |   0.8035
Max Error    |   3.02e+08   |   2.79e+08
Median AE    |   3.34e+06   |   3.40e+06



In [24]:
input_example = X_train.iloc[0:1]

for report in reports:
    with mlflow.start_run(run_name=report["name"]):
        if report["params"]:
            mlflow.log_params(report["params"])
        mlflow.log_metrics({
            "mse_val": report["mse_val"],
            "r2_val": report["r2_val"],
            "mae_val": report["mae_val"],
            "evs_val": report["evs_val"],
            "max_err_val": report["max_err_val"],
            "medae_val": report["medae_val"]
        })
        signature = infer_signature(input_example, report["model"].predict(input_example))
        if report["name"] == "XGBoost":
            mlflow.xgboost.log_model(report["model"], "model", input_example=input_example, signature=signature,  metadata={"Data Source": "Clean Train Data"})
        else:
            mlflow.sklearn.log_model(report["model"], "model", input_example=input_example, signature=signature, metadata={"Data Source": "Clean Train Data"})

2025/06/28 19:24:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/28 19:24:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: http://localhost:5000/#/experiments/306040637862002389/runs/dd4c3d0868464c6f9545e13016ed4894
🧪 View experiment at: http://localhost:5000/#/experiments/306040637862002389


2025/06/28 19:24:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://localhost:5000/#/experiments/306040637862002389/runs/8f20b112a0404e60851e008fbbdd3203
🧪 View experiment at: http://localhost:5000/#/experiments/306040637862002389


/home/elianaostro/miniconda3/lib/python3.12/site-packages/xgboost/sklearn.py:1028: UserWarning: [19:24:52] WARNING: /workspace/src/c_api/c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2025/06/28 19:24:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost at: http://localhost:5000/#/experiments/306040637862002389/runs/8a2458a502834058aa95ccb044ca8a09
🧪 View experiment at: http://localhost:5000/#/experiments/306040637862002389
🏃 View run MLP at: http://localhost:5000/#/experiments/306040637862002389/runs/51f78a1a69a0485cac848c42f018210c
🧪 View experiment at: http://localhost:5000/#/experiments/306040637862002389
